**Конвертация таблицы из docx в csv**

In [2]:
!pip install python-docx
!pip -q install pdfplumber

In [3]:
# Загрузка библиотек
from docx import Document
import pandas as pd
import numpy as np
import os
import docx
import re

import pdfplumber

In [4]:
path = 'E:/IrkutskProject/Data/'
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
original_names = {
    2016: 'soc_znach2016.docx',
    2017: 'soc_znach2017.docx',
    2018: 'soc_znach2018.docx',
    2019: 'soc_znach2019.docx',
    2020: 'soc_znach2020.docx',
    2021: '11_socialno-znachimaje-zabolev-2021.pdf',
    2022: '11_Социально_значимые_заболевания_населения_России_в_2022_году.pdf',
    2023: '11_Социально_значимые_заболевания_населения_России_в_2023_году.pdf',
    2024: '11_Социально_значимые_заболевания_населения_России_в_2024_году.pdf'
}
original_url = {}
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if not os.path.exists(original_url[year]):
        print(f"Файл {original_url[year]} не найден по указанному пути")
    else:
        print(f"Файл {original_url[year]} найден по указанному пути")

Файл E:/IrkutskProject/Data/2016/Original/soc_znach2016.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2017/Original/soc_znach2017.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2018/Original/soc_znach2018.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2019/Original/soc_znach2019.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2020/Original/soc_znach2020.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2021/Original/11_socialno-znachimaje-zabolev-2021.pdf найден по указанному пути
Файл E:/IrkutskProject/Data/2022/Original/11_Социально_значимые_заболевания_населения_России_в_2022_году.pdf найден по указанному пути
Файл E:/IrkutskProject/Data/2023/Original/11_Социально_значимые_заболевания_населения_России_в_2023_году.pdf найден по указанному пути
Файл E:/IrkutskProject/Data/2024/Original/11_Социально_значимые_заболевания_населения_России_в_2024_году.pdf найден по указанному пути


In [5]:
def doc_extract_tuberculosis_table(file_path, target_block):
    doc = docx.Document(file_path)
    target_found = False
    
    # Перебираем все элементы в теле документа
    for i, element in enumerate(doc.element.body):
        
        # Шаг 1: Ищем текст заголовка
        # Проверяем только абзацы ('p'), чтобы не искать внутри других таблиц случайно
        if not target_found and element.tag.endswith('p'):
            text = "".join(element.itertext()).strip()
            # display(text)
            if target_block.lower() in text.lower():
                target_found = True
                print(f"Текст '{target_block}' найден. Ищем следующую таблицу...")
                # continue 
        
        # Шаг 2: После того как текст найден, ищем ПЕРВУЮ встречную таблицу
        if target_found and element.tag.endswith('tbl'):
            # Считаем, сколько таблиц встретилось в документе до текущего момента
            # Это и будет порядковый индекс в doc.tables
            table_index = len([e for e in doc.element.body[:i] if e.tag.endswith('tbl')])
            
            print(f"Таблица найдена (индекс в документе: {table_index})")
            return doc.tables[table_index]

    if not target_found:
        raise ValueError(f"Текст '{target_block}' не найден в документе")
    else:
        raise ValueError(f"Текст найден, но после него нет ни одной таблицы")

def doc_table_to_dataframe(table):
    data = []
    for row in table.rows:
        current_row = []
        for cell in row.cells:
            text = cell.text.strip()
            current_row.append(text)
        data.append(current_row)
    
    # Создаем DataFrame
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

In [6]:
def pdf_extract_tuberculosis_table(pdf_path) -> list:
    all_tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            if (page_num < 6):
                continue
            elif (page_num == 6) or (page_num == 7):
                # Извлекаем таблицы с текущей страницы. page.extract_tables() возвращает список таблиц.
                # Каждая таблица — это список строк, где каждая строка — список ячеек.
                tables_on_page = page.extract_tables()
                for table_num, table_data in enumerate(tables_on_page):
                    if table_data:
                        # Первую строку таблицы можно использовать как заголовки, если она есть.
                        # Здесь для простоты берем первую строку как заголовки.
                        df = pd.DataFrame(table_data[1:], columns=table_data[0])
                        print(f"Найдена таблица на стр. {page_num + 1}, табл. {table_num + 1}, форма: {df.shape}")
                        all_tables.append(df)
            elif (page_num > 7):
                break
    return all_tables

In [7]:
def pdf_table_to_dataframe(tables):
    df_full = pd.DataFrame()
    for i, df in enumerate(tables):
        if i > 0:
            df.drop(df.index[0:2], inplace=True)
        df_full = pd.concat([df_full, df], ignore_index=True)
    return df_full

In [8]:
try:
    df = {}
    target_block = "Заболеваемость и контингенты пациентов активным туберкулезом\nпо субъектам Российской Федерации"
    # Обработка doc-файлов
    for year in years:
        display(year)
        if year < 2021:
            table = doc_extract_tuberculosis_table(original_url[year], target_block)
            df[year] = doc_table_to_dataframe(table)
        else:
            tables = pdf_extract_tuberculosis_table(original_url[year])
            df[year] = pdf_table_to_dataframe(tables)
            
        df[year].columns = [
            'СУБЪЕКТЫ ФЕДЕРАЦИИ',
            'число пациентов с впервые в жизни установ.диагнозом',
            'число пациентов с впервые в жизни установ.диагнозом',
            'число пациентов с впервые в жизни установ.диагнозом',
            'число пациентов с впервые в жизни установ.диагнозом',
            'число пациентов состоящих под диспансерным наблюдением на конец года',
            'число пациентов состоящих под диспансерным наблюдением на конец года',
            'число пациентов состоящих под диспансерным наблюдением на конец года',
            'число пациентов состоящих под диспансерным наблюдением на конец года'
        ]

        df[year].iloc[0,0] = 'СУБЪЕКТЫ ФЕДЕРАЦИИ'
        df[year].iloc[0,1] = 'абсолютные числа'
        df[year].iloc[0,2] = 'абсолютные числа'
        df[year].iloc[0,3] = 'на 100000 соот.населения'
        df[year].iloc[0,4] = 'на 100000 соот.населения'
        df[year].iloc[0,5] = 'абсолютные числа'
        df[year].iloc[0,6] = 'абсолютные числа'
        df[year].iloc[0,7] = 'на 100000 соот.населения'
        df[year].iloc[0,8] = 'на 100000 соот.населения'

        df[year]['СУБЪЕКТЫ ФЕДЕРАЦИИ'] = df[year]['СУБЪЕКТЫ ФЕДЕРАЦИИ'].str.replace(r'\n', ' ', regex=True)
        df[year].drop(df[year].index[96:], inplace=True)
        df[year].drop(df[year].index[2:3], inplace=True)
        df[year] = df[year].reset_index(drop=True)
            
        df[year].to_csv(f'E:/IrkutskProject/Data/{year}/tuberculosis_data.csv', index=False, encoding='utf-8-sig')       
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")

2016

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 4)


2017

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


2018

...

Найдена таблица на стр. 7, табл. 1, форма: (51, 9)
Найдена таблица на стр. 8, табл. 1, форма: (48, 9)


2023

Найдена таблица на стр. 7, табл. 1, форма: (50, 9)
Найдена таблица на стр. 8, табл. 1, форма: (49, 9)


2024

Найдена таблица на стр. 7, табл. 1, форма: (51, 9)
Найдена таблица на стр. 8, табл. 1, форма: (48, 9)


In [9]:
df[2016]

,СУБЪЕКТЫ ФЕДЕРАЦИИ,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года
0,СУБЪЕКТЫ ФЕДЕРАЦИИ,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения
1,СУБЪЕКТЫ ФЕДЕРАЦИИ,2015,2016,2015,2016,2015,2016,2015,2016
2,Центральный федеральный округ,14700,13375,"37,7","34,2",26988,23615,"69,0","60,4"
3,Белгородская область,420,333,"27,1","21,5",564,428,"36,4","27,6"
4,Брянская область,729,655,"59,3","53,4",1433,1389,"116,9","113,3"
...,...,...,...,...,...,...,...,...,...
90,Амурская область,621,539,"76,9","66,9",2290,2102,"284,2","260,9"
91,Магаданская область,109,82,"74,0","56,0",228,174,"155,8","118,9"
92,Сахалинская область,320,306,"65,6","62,8",1177,1109,"241,5","227,6"
93,Еврейская автономная область,209,202,"125,0","121,6",592,500,"356,4","301,0"


In [10]:
df[2021]

,СУБЪЕКТЫ ФЕДЕРАЦИИ,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года
0,СУБЪЕКТЫ ФЕДЕРАЦИИ,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения
1,None,2020,2021,2020,2021,2020,2021,2020,2021
2,Центральный федеральный округ,7667,7582,"19,5","19,3",9508,8439,"24,2","21,5"
3,Белгородская область,183,175,"11,8","11,4",235,208,"15,2","13,5"
4,Брянская область,333,279,28,"23,6",573,452,"48,4","38,2"
...,...,...,...,...,...,...,...,...,...
90,Амурская область,374,367,"47,6","46,9",1302,1212,"166,5",155
91,Магаданская область,40,43,"28,7","30,9",95,97,"68,3","69,8"
92,Сахалинская область,253,233,52,48,645,531,"132,8","109,3"
93,Еврейская автономная область,97,109,"61,6","69,6",364,326,"232,6","208,3"


In [11]:
def get_tuberculosis_dataframe (df, years, clarified_flag):
    columns = [
        'Округ',
        'Субъект',
        'Показатель',
        'Уточнение',
        'Год',
        'Значение',
        'Источник'
    ]

    data = []

    for year in years:
        for row_number in range(2, df[year].shape[0]):
            if (
                ('округ' in df[year].iloc[row_number,0]) and 
                (df[year].iloc[row_number,0] != 'Ненецкий автономный округ') and
                (df[year].iloc[row_number,0] != 'Архангельская область без автономного округа') and 
                (df[year].iloc[row_number,0] != 'Тюменская область без автономного округа') and 
                (df[year].iloc[row_number,0] != 'Чукотский автономный округ')
            ):
                district = df[year].iloc[row_number,0]
                continue

            # число пациентов с впервые в жизни установ.диагнозом + абсолютные показатели
            new_row = []
            new_row.append(district) # Округ
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[2]) # Показатель
            new_row.append(df[year].iloc[0, 2]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 1].replace(',','.')) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 2].replace(',','.')) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

            # число пациентов с впервые в жизни установ.диагнозом + на 100000 соот.населения
            new_row = []
            new_row.append(district) # Округ
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[4]) # Показатель
            new_row.append(df[year].iloc[0, 4]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 3].replace(',','.')) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 4].replace(',','.')) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

            # число пациентов состоящих под диспансерным наблюдением на конец года + абсолютные числа
            new_row = []
            new_row.append(district) # Округ
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[6]) # Показатель
            new_row.append(df[year].iloc[0, 6]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 5].replace(',','.')) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 6].replace(',','.')) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

            # число пациентов состоящих под диспансерным наблюдением на конец года + на 100000 соот.населения
            new_row = []
            new_row.append(district) # Округ
            new_row.append(df[year].iloc[row_number, 0]) # Субъект
            new_row.append(df[year].columns[8]) # Показатель
            new_row.append(df[year].iloc[0, 8]) # Уточнение
            new_row.append(year) # Год
            if (clarified_flag) and (year + 1) in original_names:
                new_row.append(df[year + 1].iloc[row_number, 7].replace(',','.')) # Значение
                new_row.append(original_names[year + 1]) # Источник
            else:
                new_row.append(df[year].iloc[row_number, 8].replace(',','.')) # Значение
                new_row.append(original_names[year]) # Источник
            data.append(new_row)

    df_long = pd.DataFrame(data, columns=columns)
    return df_long

In [12]:
df_tuberculosis_long = get_tuberculosis_dataframe (df, years, False)
df_tuberculosis_long.to_csv(f'{path}Common/tuberculosis_2016-2024.csv', index=False)

In [13]:
df_tuberculosis_long_clarified = get_tuberculosis_dataframe (df, years, True)
df_tuberculosis_long_clarified.to_csv(f'{path}Common/tuberculosis_clarified_2016-2024.csv', index=False)

In [14]:
years = [2016, 2017, 2018]
df_tuberculosis_long = get_tuberculosis_dataframe (df, years, False)
df_tuberculosis_long.to_csv(f'{path}Common_2016-2018/tuberculosis_2016-2018.csv', index=False)

df_tuberculosis_long_clarified = get_tuberculosis_dataframe (df, years, True)
df_tuberculosis_long_clarified.to_csv(f'{path}Common_2016-2018/tuberculosis_clarified_2016-2018.csv', index=False)